# 02 Data Processing

Convert weekly Google Trends data to monthly frequency, align it with monthly Census sales, and save category-level pairs.

In [ ]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd


def find_project_root() -> Path:
    cwd = Path.cwd()
    return cwd.parent if cwd.name == "notebooks" else cwd


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

trends_path = RAW_DIR / "google_trends_weekly.csv"
census_path = PROCESSED_DIR / "census_clean.csv"

if not trends_path.exists():
    raise FileNotFoundError(f"Missing {trends_path}. Run 01_data_collection.ipynb first.")
if not census_path.exists():
    raise FileNotFoundError(f"Missing {census_path}. Run 01_data_collection.ipynb first.")

trends_df = pd.read_csv(trends_path, index_col=0, parse_dates=True)
census_df = pd.read_csv(census_path, parse_dates=["date"])

print("Trends weekly shape:", trends_df.shape)
print("Census clean shape:", census_df.shape)

In [ ]:
# Resample trends from weekly to monthly using the mean.
trends_monthly = trends_df.resample("MS").mean()
trends_monthly.index.name = "date"

print("Trends monthly shape:", trends_monthly.shape)
print(trends_monthly.head())

In [ ]:
# Map search category names to Census category names.
category_map = {
    "running_shoes": "Clothing and clothing accessories stores",
    "apparel": "Clothing and clothing accessories stores",
    "furniture": "Furniture and home furnishings stores",
    "laptops": "Electronics and appliance stores",
    "washing_machines": "Electronics and appliance stores",
}

aligned_pairs = {}

for search_cat, census_cat in category_map.items():
    if search_cat not in trends_monthly.columns:
        print(f"{search_cat}: missing search column, skipping")
        continue

    search_series = trends_monthly[search_cat].copy()
    sales_series = census_df[census_df["category"] == census_cat].set_index("date")["sales_millions"]

    combined = pd.DataFrame({
        "search": search_series,
        "sales": sales_series,
    }).dropna()

    aligned_pairs[search_cat] = combined
    print(f"{search_cat}: {len(combined)} months of aligned data")

In [ ]:
aligned_pairs_path = PROCESSED_DIR / "aligned_pairs.pkl"

with open(aligned_pairs_path, "wb") as f:
    pickle.dump(aligned_pairs, f)

print("Saved:", aligned_pairs_path)
print("Aligned categories:", sorted(aligned_pairs))